# 01b Data Cleaning - Standardized Input

## 1.1 Purpose

This notebook is a standardized-input version of the original data cleaning workflow. It is prepared for future company sales datasets that have already been mapped into the normalized schema by `scripts/standardize_raw_sales.py`.

This notebook is a preparation layer, not a replacement for `01_data_cleaning.ipynb` yet. It writes to `data/processed_standardized/` so the current validated project outputs in `data/processed/` are not overwritten.

## 1.2 Expected Input

The notebook consumes:

```text
data/interim/standardized_sales.csv
```

Expected normalized columns:

- `invoice_no`
- `stock_code`
- `description`
- `quantity`
- `invoice_date`
- `unit_price`
- `customer_id`
- `country`

`stock_code` is used as the normalized SKU key. `description` is treated as a display field so that minor description variations do not split the same SKU into multiple profiles.

## 1.3 Cleaning Strategy

The cleaning process mirrors notebook 01 using the normalized column names:

1. Remove exact duplicate rows.
2. Separate cancellations and returns.
3. Remove invalid sales records with non-positive quantity or unit price.
4. Exclude records without product descriptions.
5. Retain missing customer IDs when the sales transaction is otherwise valid.
6. Exclude non-product transaction lines such as postage, fees, discounts, bank charges, and manual adjustments.
7. Create SKU-level and monthly SKU-level outputs for downstream analysis.

In [1]:
import pandas as pd
from pathlib import Path

standardized_path = Path("../data/interim/standardized_sales.csv")
processed_dir = Path("../data/processed_standardized")
processed_dir.mkdir(parents=True, exist_ok=True)

if not standardized_path.exists():
    raise FileNotFoundError(
        "Standardized input not found. Run scripts/standardize_raw_sales.py first. "
        f"Expected file: {standardized_path}"
    )

df = pd.read_csv(
    standardized_path,
    dtype={
        "invoice_no": "string",
        "stock_code": "string",
        "description": "string",
        "customer_id": "string",
        "country": "string",
    },
)

print("Standardized input shape:", df.shape)
df.head()

Standardized input shape: (541909, 8)


,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## 1.4 Validate Standardized Schema

This step checks that the expected normalized fields are available before cleaning begins. The schema mapping and validation scripts should normally catch these issues earlier, but this check keeps the notebook self-explanatory and safe to run independently.

In [2]:
expected_columns = [
    "invoice_no",
    "stock_code",
    "description",
    "quantity",
    "invoice_date",
    "unit_price",
    "customer_id",
    "country",
]

missing_columns = sorted(set(expected_columns) - set(df.columns))
if missing_columns:
    raise ValueError("Missing standardized input columns: " + ", ".join(missing_columns))

df = df[expected_columns].copy()

df["invoice_no"] = df["invoice_no"].astype("string").str.strip()
df["stock_code"] = df["stock_code"].astype("string").str.strip()
df["description"] = df["description"].astype("string")
df["customer_id"] = df["customer_id"].astype("string").str.strip()
df["country"] = df["country"].astype("string").str.strip()
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")
df["invoice_date"] = pd.to_datetime(df["invoice_date"], errors="coerce")

print("Missing values:")
print(df.isna().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

Missing values:
invoice_no           0
stock_code           0
description       1454
quantity             0
invoice_date         0
unit_price           0
customer_id     135080
country              0
dtype: int64

Duplicate rows:
5268

Data types:
invoice_no              string
stock_code              string
description             string
quantity                 int64
invoice_date    datetime64[us]
unit_price             float64
customer_id             string
country                 string
dtype: object


## 1.5 Clean Valid Sales Transactions

The main sales dataset should represent valid physical product demand. Cancellations and returns are saved separately, while valid sales records are retained only when quantity, unit price, product code, product description, and invoice date are usable.

In [3]:
df_clean = df.drop_duplicates().copy()

returns_cancellations = df_clean[
    df_clean["invoice_no"].str.startswith("C", na=False)
    | (df_clean["quantity"] < 0)
].copy()

sales = df_clean[
    (~df_clean["invoice_no"].str.startswith("C", na=False))
    & (df_clean["quantity"] > 0)
    & (df_clean["unit_price"] > 0)
    & (df_clean["stock_code"].notna())
    & (df_clean["description"].notna())
    & (df_clean["description"].str.strip() != "")
    & (df_clean["invoice_date"].notna())
].copy()

sales["description"] = sales["description"].str.strip().str.upper()
sales["stock_code"] = sales["stock_code"].str.strip()
sales["country"] = sales["country"].str.strip()

sales["invoice_date_only"] = sales["invoice_date"].dt.date
sales["invoice_month"] = sales["invoice_date"].dt.to_period("M").astype(str)
sales["invoice_week"] = sales["invoice_date"].dt.to_period("W").astype(str)
sales["revenue"] = sales["quantity"] * sales["unit_price"]

print("Rows after removing duplicates:", len(df_clean))
print("Valid sales rows before excluding non-product lines:", len(sales))
print("Returns / cancellations rows:", len(returns_cancellations))

sales.head()

Rows after removing duplicates: 536641
Valid sales rows before excluding non-product lines: 524878
Returns / cancellations rows: 10587


,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,invoice_date_only,invoice_month,invoice_week,revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,2010-12-01,2010-12,2010-11-29/2010-12-05,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,2010-12-01,2010-12,2010-11-29/2010-12-05,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,2010-12-01,2010-12,2010-11-29/2010-12-05,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,2010-12-01,2010-12,2010-11-29/2010-12-05,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,2010-12-01,2010-12,2010-11-29/2010-12-05,20.34


## 1.6 Exclude Non-Product Transaction Lines

Some transaction lines are not physical product sales. These include postage, service fees, bank charges, discounts, marketplace fees, charity-related lines, and manual adjustments. These records are saved separately for transparency and excluded from the main product sales dataset.

In [4]:
non_product_stock_codes = [
    "POST",        # Postage
    "DOT",         # Dotcom postage / service-related line
    "BANK CHARGES",
    "AMAZONFEE",
    "CRUK",
    "D",           # Discount
    "M",           # Manual adjustment
]

non_product_lines = sales[sales["stock_code"].isin(non_product_stock_codes)].copy()
sales = sales[~sales["stock_code"].isin(non_product_stock_codes)].copy()

print("Non-product transaction rows excluded:", len(non_product_lines))
print("Product sales rows after excluding non-product lines:", len(sales))

non_product_lines.head()

Non-product transaction rows excluded: 2162
Product sales rows after excluding non-product lines: 522716


,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,invoice_date_only,invoice_month,invoice_week,revenue
45,536370,POST,POSTAGE,3,2010-12-01 08:45:00,18.00,12583.0,France,2010-12-01,2010-12,2010-11-29/2010-12-05,54.00
386,536403,POST,POSTAGE,1,2010-12-01 11:27:00,15.00,12791.0,Netherlands,2010-12-01,2010-12,2010-11-29/2010-12-05,15.00
1123,536527,POST,POSTAGE,1,2010-12-01 13:04:00,18.00,12662.0,Germany,2010-12-01,2010-12,2010-11-29/2010-12-05,18.00
1814,536544,DOT,DOTCOM POSTAGE,1,2010-12-01 14:32:00,569.77,<NA>,United Kingdom,2010-12-01,2010-12,2010-11-29/2010-12-05,569.77
2239,536569,M,MANUAL,1,2010-12-01 15:35:00,1.25,16274.0,United Kingdom,2010-12-01,2010-12,2010-11-29/2010-12-05,1.25


## 1.7 SKU Master Table

The SKU master table uses `stock_code` as the normalized SKU key. Product descriptions are retained as display fields and supporting context, not as the unique SKU identifier.

In [5]:
sku_master = (
    sales
    .sort_values("invoice_date")
    .groupby("stock_code", as_index=False)
    .agg(
        description=("description", "first"),
        first_sale_date=("invoice_date", "min"),
        last_sale_date=("invoice_date", "max"),
        total_units=("quantity", "sum"),
        total_revenue=("revenue", "sum"),
        avg_unit_price=("unit_price", "mean"),
    )
)

print("SKU master shape:", sku_master.shape)
sku_master.head()

SKU master shape: (3917, 7)


,stock_code,description,first_sale_date,last_sale_date,total_units,total_revenue,avg_unit_price
0,10002,INFLATABLE POLITICAL GLOBE,2010-12-01 08:45:00,2011-04-18 12:56:00,860,759.89,1.086620
1,10080,GROOVY CACTUS INFLATABLE,2011-02-27 13:47:00,2011-11-21 17:04:00,303,119.09,0.410909
2,10120,DOGGY RUBBER,2010-12-03 11:19:00,2011-12-04 13:15:00,192,40.32,0.210000
3,10123C,HEARTS WRAPPING TAPE,2010-12-03 11:19:00,2011-03-31 13:14:00,5,3.25,0.650000
4,10124A,SPOTS ON RED BOOKCOVER TAPE,2010-12-06 13:13:00,2011-11-06 13:00:00,16,6.72,0.420000


## 1.8 SKU Description Consistency Check

This table checks whether a single `stock_code` appears with multiple descriptions. The downstream SKU profile still uses `stock_code` as the key; description variation is treated as a data quality and reporting issue.

In [6]:
sku_description_check = (
    sales
    .groupby("stock_code", as_index=False)
    .agg(
        description_count=("description", "nunique"),
        total_units=("quantity", "sum"),
        total_revenue=("revenue", "sum"),
    )
    .sort_values("description_count", ascending=False)
)

sku_description_check.to_csv(processed_dir / "sku_description_check.csv", index=False)
sku_description_check.head(20)

,stock_code,description_count,total_units,total_revenue
2089,23236,4,2493,7059.97
2049,23196,4,1872,2842.24
2245,23413,3,173,909.31
2056,23203,3,20485,42030.34
2203,23366,3,2249,1667.95
2084,23231,3,7277,3008.34
104,17107D,3,170,433.50
2093,23240,3,4177,17000.17
1802,22937,3,972,2563.24
1980,23126,3,1109,5532.04


## 1.9 Monthly SKU-Level Sales Table

The cleaned transaction-level sales records are aggregated to `stock_code` and `invoice_month`. This creates a monthly SKU-level dataset that can support future classification, replenishment, and inventory analytics once the standardized pipeline is connected downstream.

In [7]:
monthly_sku_sales = (
    sales
    .groupby(["stock_code", "invoice_month"], as_index=False)
    .agg(
        monthly_units=("quantity", "sum"),
        monthly_revenue=("revenue", "sum"),
        order_count=("invoice_no", "nunique"),
        avg_unit_price=("unit_price", "mean"),
    )
)

monthly_sku_sales = monthly_sku_sales.merge(
    sku_master[["stock_code", "description"]],
    on="stock_code",
    how="left",
)

monthly_sku_sales = monthly_sku_sales[
    [
        "stock_code",
        "description",
        "invoice_month",
        "monthly_units",
        "monthly_revenue",
        "order_count",
        "avg_unit_price",
    ]
]

monthly_sku_sales.to_csv(processed_dir / "monthly_sku_sales.csv", index=False)

print("Monthly SKU sales shape:", monthly_sku_sales.shape)
monthly_sku_sales.head()

Monthly SKU sales shape: (34020, 7)


,stock_code,description,invoice_month,monthly_units,monthly_revenue,order_count,avg_unit_price
0,10002,INFLATABLE POLITICAL GLOBE,2010-12,251,234.41,30,1.201000
1,10002,INFLATABLE POLITICAL GLOBE,2011-01,340,291.37,21,0.962857
2,10002,INFLATABLE POLITICAL GLOBE,2011-02,52,45.76,7,1.072857
3,10002,INFLATABLE POLITICAL GLOBE,2011-03,28,27.70,8,1.142500
4,10002,INFLATABLE POLITICAL GLOBE,2011-04,189,160.65,5,0.850000


## 1.10 Save Cleaned Outputs

All outputs are written to `data/processed_standardized/`. This keeps the standardized-input preparation workflow separate from the current validated outputs in `data/processed/`.

In [8]:
sales.to_csv(processed_dir / "clean_sales.csv", index=False)
returns_cancellations.to_csv(processed_dir / "returns_cancellations.csv", index=False)
non_product_lines.to_csv(processed_dir / "non_product_lines.csv", index=False)
sku_master.to_csv(processed_dir / "sku_master.csv", index=False)

print("Saved files:")
print("- data/processed_standardized/clean_sales.csv")
print("- data/processed_standardized/returns_cancellations.csv")
print("- data/processed_standardized/non_product_lines.csv")
print("- data/processed_standardized/sku_master.csv")
print("- data/processed_standardized/sku_description_check.csv")
print("- data/processed_standardized/monthly_sku_sales.csv")

Saved files:
- data/processed_standardized/clean_sales.csv
- data/processed_standardized/returns_cancellations.csv
- data/processed_standardized/non_product_lines.csv
- data/processed_standardized/sku_master.csv
- data/processed_standardized/sku_description_check.csv
- data/processed_standardized/monthly_sku_sales.csv


## 1.11 Data Quality Summary

The data quality summary documents the row-level transformation from standardized input to final cleaned product sales records.

In [9]:
data_quality_summary = pd.DataFrame({
    "step": [
        "standardized_input",
        "after_duplicate_removal",
        "returns_cancellations_separated",
        "non_product_lines_excluded",
        "valid_product_sales_final",
        "monthly_sku_sales_records",
        "sku_master_records",
    ],
    "row_count": [
        len(df),
        len(df_clean),
        len(returns_cancellations),
        len(non_product_lines),
        len(sales),
        len(monthly_sku_sales),
        len(sku_master),
    ],
    "note": [
        "Original standardized transaction rows from data/interim/standardized_sales.csv",
        "Rows after removing exact duplicate records",
        "Cancellation and return-related records separated from main sales data",
        "Postage, fees, discounts, bank charges, manual adjustments, and other non-product lines excluded",
        "Final valid physical product sales records",
        "Monthly SKU-level aggregation records",
        "Normalized SKU master records using stock_code as SKU key",
    ],
})

data_quality_summary.to_csv(processed_dir / "data_quality_summary.csv", index=False)
data_quality_summary

,step,row_count,note
0,standardized_input,541909,Original standardized transaction rows from da...
1,after_duplicate_removal,536641,Rows after removing exact duplicate records
2,returns_cancellations_separated,10587,Cancellation and return-related records separa...
3,non_product_lines_excluded,2162,"Postage, fees, discounts, bank charges, manual..."
4,valid_product_sales_final,522716,Final valid physical product sales records
5,monthly_sku_sales_records,34020,Monthly SKU-level aggregation records
6,sku_master_records,3917,Normalized SKU master records using stock_code...


## 1.12 Final Output Summary

This notebook creates a parallel set of standardized-input outputs:

1. `clean_sales.csv`
2. `returns_cancellations.csv`
3. `non_product_lines.csv`
4. `sku_master.csv`
5. `sku_description_check.csv`
6. `monthly_sku_sales.csv`
7. `data_quality_summary.csv`

These files are intentionally stored under `data/processed_standardized/` and do not replace the current project outputs.

In [10]:
print("===== 01b Standardized-Input Data Cleaning Summary =====")
print("Valid product sales rows:", len(sales))
print("Returns / cancellations rows:", len(returns_cancellations))
print("Non-product transaction rows excluded:", len(non_product_lines))
print("Monthly SKU sales shape:", monthly_sku_sales.shape)
print("SKU master shape:", sku_master.shape)

print("\nOutput files:")
print("- data/processed_standardized/clean_sales.csv")
print("- data/processed_standardized/returns_cancellations.csv")
print("- data/processed_standardized/non_product_lines.csv")
print("- data/processed_standardized/sku_master.csv")
print("- data/processed_standardized/monthly_sku_sales.csv")
print("- data/processed_standardized/data_quality_summary.csv")
print("- data/processed_standardized/sku_description_check.csv")

===== 01b Standardized-Input Data Cleaning Summary =====
Valid product sales rows: 522716
Returns / cancellations rows: 10587
Non-product transaction rows excluded: 2162
Monthly SKU sales shape: (34020, 7)
SKU master shape: (3917, 7)

Output files:
- data/processed_standardized/clean_sales.csv
- data/processed_standardized/returns_cancellations.csv
- data/processed_standardized/non_product_lines.csv
- data/processed_standardized/sku_master.csv
- data/processed_standardized/monthly_sku_sales.csv
- data/processed_standardized/data_quality_summary.csv
- data/processed_standardized/sku_description_check.csv
